In [6]:
class SimpleGradebook(object):
    def __init__(self):
        self._grades = {}
    
    def add_student(self,name):
        self._grades[name] = {}
        
    def report_grade(self, name, subject, grade):
        by_subject = self._grades[name]
        grade_list = by_subject.setdefault(subject,[])
        grade_list.append(grade)
    
    def average_grade(self, name):
        by_subject = self._grades[name]
        total, count = 0,0
        for grades in by_subject.values():
            total += sum(grades)
            count += len(grades)
        return total /count
    
book = SimpleGradebook()
book.add_student('Issac Newton')
book.report_grade('Issac Newton','수학',90)
book.report_grade('Issac Newton','과학',80)
print(book.average_grade('Issac Newton'))

85.0


In [12]:
import collections
Grade = collections.namedtuple('Grade',('score','weight'))
## namedtuple을 사용하자

class Subject(object):
    def __init__(self):
        self._grades = []
    
    def report_grade(self, score, weight):
        self._grades.append(Grade(score,weight))
        
    def average_grade(self):
        total, total_weight =0,0
        for grade in self._grades:
            total += grade.score * grade.weight
            total_weight += grade.weight
            
        return total/total_weight
    
class Student(object):
    def __init__(self):
        self._subjects ={}
        
    def subject(self,name):
        if name not in self._subjects:
            self._subjects[name] = Subject()
        return self._subjects[name]
    
    def average_grade(self):
        total, count =0,0
        for subject in self._subjects.values():
            total += subject.average_grade()
            count += 1
        return total / count

class GradeBook(object):
    def __init__(self):
        self._students = {}
    
    def student(self, name):
        if name not in self._students:
            self._students[name] = Student()
        return self._students[name]
    
book=GradeBook()
albert = book.student("알버트 아인슈타인")
math = albert.subject('Math')
math.report_grade(80, 0.10)
eng = albert.subject('English')
eng.report_grade(90, 0.20)
print(albert.average_grade())

85.0


In [14]:
names = ['엄엄엄엄','김칠복','엄칠복','너굴너굴이']
names.sort(key=lambda x: len(x))
print(names)

['김칠복', '엄칠복', '엄엄엄엄', '너굴너굴이']


In [16]:
from collections import defaultdict
def log_missing():
    print('Key added')
    return 0

current = {'green':12,'blue':3}
increments = [
    ('red',5),
    ('blue',17),
    ('orange',9)
]
result = defaultdict(log_missing, current)
print('Before:', dict(result))
for key, amount in increments:
    result[key] += amount
print('After:', dict(result))

Before: {'green': 12, 'blue': 3}
Key added
Key added
After: {'green': 12, 'blue': 20, 'red': 5, 'orange': 9}


In [19]:
current = {'green':12,'blue':3}
increments = [
    ('red',5),
    ('blue',17),
    ('orange',9)
]

def increment_with_report(current, increments):
    added_count = 0
    
    
    def missing():
        nonlocal added_count
        added_count +=1
        return 0
    result = defaultdict(missing,current)
    for key, amount in increments:
        result[key] += amount
    
    return result, added_count

result, count = increment_with_report(current, increments)
assert count == 2

In [20]:
from collections import defaultdict

current = {'green':12,'blue':3}
increments = [
    ('red',5),
    ('blue',17),
    ('orange',9)
]

class CountMissing(object):
    def __init__(self):
        self.added =0
        
    def missing(self):
        self.added +=1
        return 0
    
counter = CountMissing()
result = defaultdict(counter.missing, current)

for key, amount in increments:
    result[key] += amount
assert counter.added == 2

In [26]:
from collections import defaultdict



class BetterCounterMissing(object):
    def __init__(self):
        self.added = 0
    def __call__(self):
        self.added +=1
        return 0
    
counter = BetterCounterMissing()
counter()
assert callable(counter)
result = defaultdict(counter,current)
for key, amount in increments:
    result[key] += amount
assert counter.added ==3


In [ ]:
## Read메서드가 있는 입력 데이터 클래스
class InputData(object):
    def read(self):
        raise NotImplementedError
    
## Disk file에서 데이터를 읽어오도록 구현한 InputData의 서브 클래스
class PathInputData(InputData):
    def __init__(self, path):
        super().__init__()
        self.path = path
    def read(self):
        return open(self.path).read()
    
## 작업 추상화 클래스
class Worker(object):
    def __init__(self, input_data):
        self.input_data = input_data
        self.result = None
    
    def map(self):
        raise NotImplementedError
    
    def reduce(self, other):
        raise NotImplementedError
## Worker 구쳇 서브 클래스    
class LineCountWorker(Worker):
    def map(self):
        data = self.input_data.read()
        self.result = data.count('\n')
        
    def reduce(self,other):
        self.result += other.result
        
        
        

In [ ]:
import os
from tempfile import TemporaryDirectory
from threading import Thread

class InputData(object):
    def read(self):
        raise NotImplementedError
    
## Disk file에서 데이터를 읽어오도록 구현한 InputData의 서브 클래스
class PathInputData(InputData):
    def __init__(self, path):
        super().__init__()
        self.path = path
    def read(self):
        return open(self.path).read()
    
## 작업 추상화 클래스
class Worker(object):
    def __init__(self, input_data):
        self.input_data = input_data
        self.result = None
    
    def map(self):
        raise NotImplementedError
    
    def reduce(self, other):
        raise NotImplementedError
## Worker 구체 서브 클래스    
class LineCountWorker(Worker):
    def map(self):
        data = self.input_data.read()
        self.result = data.count('\n')
        
    def reduce(self,other):
        self.result += other.result


def generate_inputs(data_dir):
    for name in os.listdir(data_dir):
        yield PathInputData(os.path.join(data_dir, name))
        
def create_workers(input_list):
    workers =[]
    for input_data in input_list:
        workers.append(LineCountWorker(input_data))
    return workers

def execute(workers):
    threads = [Thread(target=w.map)for w in workers]
    for thread in threads: thread.start()
    for thread in threads: thread.join()
    
    first, rest = workers[0], workers[1:]
    for worker in rest:
        first.reduce(worker)
    return first.result

def mapredcue(data_dir):
    inputs = generate_inputs(data_dir)
    workers = create_workers(inputs)
    return execute(workers)

def write_test_files(tmpdir):
    test_data = {
        "file1.txt": "apple\nbanana\ncherry\n",            # 3줄
        "file2.txt": "dog\ncat\n",                         # 2줄
        "file3.txt": "python\nmapreduce\nthread\ngil\n"    # 4줄 (합계 9줄)
    }
    
    for filename, content in test_data.items():
        filepath = os.path.join(tmpdir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
            
with TemporaryDirectory() as tmpdir:
    write_test_files(tmpdir)
    result = mapredcue(tmpdir)
    
print('There are',result, 'lines')

There are 9 lines


In [ ]:
import os
from tempfile import TemporaryDirectory
from threading import Thread

    
## 공통인터페이스의 확장 : 새 inputData 인스턴스 생성 하는 범용 클래스 메서드           
class GenericInputData(object):
    def read(self):
        raise NotImplementedError

    @classmethod
    def generate_inputs(cls, config):
        raise NotImplementedError
    
## Disk file에서 데이터를 읽어오도록 구현한 InputData의 서브 클래스
class PathInputData(GenericInputData):
    def __init__(self, path):
        super().__init__()
        self.path = path
    def read(self):
        return open(self.path).read()
    
    ## 클래스 메서드
    @classmethod
    def generate_inputs(cls,config):
        data_dir = config['data_dir']
        for name in os.listdir(data_dir):
            yield cls(os.path.join(data_dir, name))
    
## 작업 추상화 클래스
class GenericWorker(object):
    def __init__(self, input_data):
        self.input_data = input_data
        self.result = None
    
    def map(self):
        raise NotImplementedError
    
    def reduce(self, other):
        raise NotImplementedError
    
    @classmethod
    def create_workers(cls, input_list, config):
        workers =[]
        for input_data in input_list.generate_inputs(config):
            workers.append(cls(input_data))
        return workers
    
## Worker 구체 서브 클래스    
class LineCountWorker(GenericWorker):
    def map(self):
        data = self.input_data.read()
        self.result = data.count('\n')
        
    def reduce(self,other):
        self.result += other.result
        

# def generate_inputs(data_dir):
#     for name in os.listdir(data_dir):
#         yield PathInputData(os.path.join(data_dir, name))
        


def execute(workers):
    threads = [Thread(target=w.map)for w in workers]
    for thread in threads: thread.start()
    for thread in threads: thread.join()
    
    first, rest = workers[0], workers[1:]
    for worker in rest:
        first.reduce(worker)
    return first.result

def mapredcue(worker_class, input_class, config):
    workers = worker_class.create_workers(input_class, config)
    return execute(workers)

def write_test_files(tmpdir):
    test_data = {
        "file1.txt": "apple\nbanana\ncherry\n",            # 3줄
        "file2.txt": "dog\ncat\n",                         # 2줄
        "file3.txt": "python\nmapreduce\nthread\ngil\n"    # 4줄 (합계 9줄)
    }
    
    for filename, content in test_data.items():
        filepath = os.path.join(tmpdir, filename)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
            
with TemporaryDirectory() as tmpdir:
    write_test_files(tmpdir)
    config = {'data_dir': tmpdir}
    result = mapredcue(LineCountWorker, PathInputData, config)
    
print('There are',result, 'lines')

There are 9 lines


In [ ]:
from pprint import pprint


class MyBaseClass(object):
    def __init__(self,value):
        self.value = value
    
class TimesFive(MyBaseClass):
    def __init__(self,value):
        super(TimesFive,self).__init__(value)
        self.value *=5
        
class PlusTwo(MyBaseClass):
    def __init__(self,value):
        super(PlusTwo,self).__init__(value)
        self.value+=2
        
class ThisWay(TimesFive, PlusTwo):
    def __init__(self,value):
        super(ThisWay,self).__init__(value)

foo = ThisWay(5)
print(foo.value)
        
pprint(ThisWay.mro())


35
